In [1]:
# 1. Загрузка сырого слоя
import pandas as pd
from pathlib import Path

RAW = Path("ege_all.csv")        # если ноутбук в notebooks/ → Path("..") / "ege_all.csv"
df = pd.read_csv(RAW, sep=";", encoding="utf-8-sig", low_memory=False)
assert df.shape == (167974, 25), df.shape
df.head(3)

,category,year,admission_type,university,field_group,region,profile,avg_score,score_change,min_score,...,enrolled_competition,enrolled_bvi,enrolled_quota,enrolled_target,enrolled_crimea,enrolled_budget,n_known_scores,tuition,id_deducted,source_file
0,Данные по направлениям подготовки,2011,Бюджетный прием,Моск. физико-техн. ин-т. (МФТИ),Математика,Москва и Московская область,NaN,89.4,NaN,79.7,...,75.0,13.0,2.0,NaN,NaN,NaN,75.0,NaN,NaN,napravleniya_2011_budget.csv
1,Данные по направлениям подготовки,2011,Бюджетный прием,Моск. гос. ун-т. им. М.В. Ломоносова,Математика,Москва и Московская область,NaN,87.6,NaN,70.7,...,577.0,148.0,29.0,NaN,NaN,NaN,577.0,NaN,NaN,napravleniya_2011_budget.csv
2,Данные по направлениям подготовки,2011,Бюджетный прием,"Национальный исследовательский ун-т. ""Высшая ш...",Математика,Москва и Московская область,NaN,85.2,NaN,80.0,...,78.0,52.0,2.0,NaN,NaN,NaN,78.0,NaN,NaN,napravleniya_2011_budget.csv


In [2]:
# 2. Осмотр
print(df.dtypes)
print("\nПропуски по колонкам:\n", df.isna().sum())
print("\nСтроки по группам:\n", df.groupby(["category", "admission_type"]).size())

category                        str
year                          int64
admission_type                  str
university                      str
field_group                     str
region                          str
profile                         str
avg_score                   float64
score_change                float64
min_score                   float64
avg_score_competition       float64
score_change_competition    float64
avg_score_no_id             float64
avg_score_budget            float64
enrolled                    float64
enrolled_competition        float64
enrolled_bvi                float64
enrolled_quota              float64
enrolled_target             float64
enrolled_crimea             float64
enrolled_budget             float64
n_known_scores              float64
tuition                     float64
id_deducted                  object
source_file                     str
dtype: object

Пропуски по колонкам:
 category                         0
year                       

In [4]:
# 3. Документированная очистка (все шаги проверены, см. DATA_NOTES.md)
BROKEN = ["vuzy_2012_paid.csv", "vuzy_2013_paid.csv"]

df["id_deducted"] = (df["id_deducted"].astype(str).str.strip()
                     .map({"True": True, "False": False}).astype("boolean"))
df.loc[df["tuition"] < 2000, "tuition"] = pd.NA          # п.2
df["qc_unreliable"] = df["source_file"].isin(BROKEN)      # п.3
df.loc[df["qc_unreliable"], ["min_score", "enrolled"]] = pd.NA
df["coverage"] = df["year"].map(lambda y: "state_only" if y <= 2017 else "all")  # п.1

assert len(df) == 167974
assert df["qc_unreliable"].sum() == 461
assert df["tuition"].min() >= 2000

In [5]:
# 4. Диагностика названий — с этого начнём нормализацию
v = df[df.category == "Данные по вузам"]
n_years = v.groupby("university")["year"].nunique()
print("уникальных названий:", len(n_years))
print("вузов во все 15 лет:", (n_years == 15).sum())
print(n_years.sort_values(ascending=False).head(10))

уникальных названий: 1385
вузов во все 15 лет: 155
university
Адыгейский гос. ун-т., г. Майкоп                                                        15
Дагестанский гос. педагогический ун-т., г. Махачкала                                    15
Дальневосточный гос. аграрный ун-т., г. Благовещенск                                    15
Северо-Кавказский горно-металлургический ин-т. (гос. технол. ун-т.) , г. Владикавказ    15
Дальневосточный гос. медицинский ун-т., г. Хабаровск                                    15
Дальневосточный гос. техн. рыбохозяйственный ун-т., г. Владивосток                      15
Дальневосточный гос. ун-т. путей сообщения, г. Хабаровск                                15
Северный гос. медицинский ун-т., г. Архангельск                                         15
Сахалинский гос. ун-т., г. Южно-Сахалинск                                               15
Саратовский гос. медицинский ун-т.                                                      15
Name: year, dtype: int64


In [6]:
# 5. Диагностика: какие именно варианты написания есть в данных
u = pd.Series(df["university"].unique())
print("уникальных названий во всём датасете:", len(u))
print("с 'филиал':", u.str.contains("филиал", case=False).sum())
print("с аббревиатурой в скобках:", u.str.contains(r"\([А-ЯЁA-Z]{3,}", regex=True).sum())
print("с хвостом ', г. ...':", u.str.contains(r",\s*г\.", regex=True).sum())
print("с правовой формой:", u.str.contains(r"ФГ[АБКО]ОУ|ГОУ ВПО|образовательное учреждение", regex=True).sum())
print("с двойными пробелами:", u.str.contains("  ", regex=False).sum())

s_v = set(df.loc[df.category == "Данные по вузам", "university"])
s_n = set(df.loc[df.category != "Данные по вузам", "university"])
print("названий только в 'направлениях':", len(s_n - s_v))   # маппинг строим по обоим разделам

уникальных названий во всём датасете: 1393
с 'филиал': 492
с аббревиатурой в скобках: 71
с хвостом ', г. ...': 348
с правовой формой: 0
с двойными пробелами: 0
названий только в 'направлениях': 8


In [9]:
# 6. Детерминированный ключ названия (проверено на реальных строках источника)
import re

LEGAL_PATTERNS = [   # правовые формы — не часть идентичности вуза, менялись по годам
    r"федеральное государственное \w+ образовательное учреждение высшего (?:профессионального )?образования",
    r"(?:не)?государственное образовательное учреждение высшего профессионального образования",
    r"фг[абко]оу(?:\s+в(?:по|о))?", r"гоу впо", r"\bаноо?\b", r"\bноу впо\b", r"\bчоу впо\b",
]
EXTRA = [(r"\bс\.-петерб\.?", "санкт-петербургский"), (r"\bс-петерб\.?", "санкт-петербургский")]
ABBR = {"моск": "московский", "гос": "государственный", "ун-т": "университет",
        "ин-т": "институт", "акад": "академия", "техн": "технический",
        "технол": "технологический", "пед": "педагогический", "мед": "медицинский",
        "нац": "национальный", "исслед": "исследовательский", "им": "имени"}
# статусные слова: "государственный", "НИУ" появились/исчезали по годам — в ключ не берём
STOP = {"государственный", "национальный", "исследовательский", "имени", "ниу"}
STATUS_PAREN = {"государственный", "национальный", "исследовательский", "университет"}
ACR_RE = re.compile(r"\(([А-ЯЁA-Z][А-ЯЁа-яA-Z0-9\-]{2,14})\)")

def _is_acr(t):
    return sum(c.isupper() for c in t) >= 3

def extract_abbr(name):
    """Аббревиатура в скобках — отдельный сигнал, но НЕ ключ (МГУ ≠ МГУ им. Огарёва)."""
    c = [t for t in ACR_RE.findall(name) if _is_acr(t)]
    return c[-1] if c else None

def _strip_status_paren(m):
    # "(национальный исследовательский университет)" у МФТИ — статус, не имя
    toks = re.findall(r"[а-я]+", m.group(1))
    return " " if toks and set(toks) <= STATUS_PAREN else m.group(0)

def make_key(name):
    s = name.replace("\xa0", " ")
    s = ACR_RE.sub(lambda m: " " if _is_acr(m.group(1)) else m.group(0), s)  # (МФТИ) → отдельно
    s = s.lower().replace("ё", "е")
    s = re.sub(r"[«»\"“”„']", " ", s)
    s = re.sub(r"[–—−]", "-", s)
    for p in LEGAL_PATTERNS:
        s = re.sub(p, " ", s)
    if "филиал" not in s:   # у филиалов город — часть идентичности, не убираем!
        s = re.sub(r"\s*,?\s*\(?\s*г\.\s*[а-яa-z][а-яa-z\- ]*\)?\s*$", " ", s)
    for p, r in EXTRA:
        s = re.sub(p, r, s)
    for stem, full in ABBR.items():
        s = re.sub(rf"\b{re.escape(stem)}\b\.?", full, s)
    s = re.sub(r"\(([^()]*)\)", _strip_status_paren, s)
    s = re.sub(r"[^а-яa-z0-9]+", " ", s)
    return " ".join(t for t in s.split() if len(t) > 1 and t not in STOP)  # len>1 — инициалы "М.В."

# смоук-тесты на реальных названиях
assert make_key("Моск. физико-техн. ин-т. (МФТИ)") == make_key(
    "Московский физико-технический институт (национальный исследовательский университет)")
assert make_key("Моск. гос. ун-т. им. М.В. Ломоносова") == make_key(
    "Московский государственный университет имени М.В. Ломоносова")
assert make_key("Сахалинский гос. ун-т., г. Южно-Сахалинск") == make_key("Сахалинский государственный университет")
assert make_key("С.-Петерб. гос. ун-т.") == make_key("Санкт-Петербургский государственный университет")
assert make_key("Моск. гос. ун-т. им. М.В. Ломоносова") != make_key("Мордовский гос. ун-т. им. Н.П. Огарёва")
kb = make_key('Национальный исследовательский ун-т.  "Высшая школа экономики", филиал, г. Пермь')
kh = make_key('Национальный исследовательский ун-т.  "Высшая школа экономики"')
assert "пермь" in kb.split() and "филиал" in kb.split()   # город и признак филиала сохранены
assert kb != kh                                           # филиал ≠ головной вуз
assert make_key("Филиал НИУ ВШЭ в г. Перми") != make_key("НИУ «Высшая школа экономики»")
assert extract_abbr("Моск. физико-техн. ин-т. (МФТИ)") == "МФТИ"
print("смоук-тесты пройдены")

смоук-тесты пройдены


In [20]:
# 7. Карта названий: авто-группировка + ручные правки, применение, проверки
MANUAL = {
    # "точное исходное название": "каноническое название",  ← заполним после ячейки 8
    'Белгородский гос. ин-т. культуры и искусств': 'Белгородский гос. ин-т. искусств и культуры',
    'Белгородский гос. национальный исследовательский ун-т. (НИУ «БелГУ»)': 'Белгородский гос. национальный исследовательский ун-т.',
    'Моск. гос. гуманитарный экономический ун-т.': 'Моск. гос. гуманитарно-экономический ун-т.',
    'Моск. гос. машиностроительный ун-т. "МАМИ" (бывший Моск. гос. техн. ун-т. "МАМИ")': 'Моск. гос. машиностроительный ун-т. (МАМИ)',
    'Моск. гос. техн. ун-т. "МАМИ"': 'Моск. гос. машиностроительный ун-т. (МАМИ)',
    'Моск. энергетический ин-т. (МЭИ)': 'Национальный исследовательский ун-т. "Моск. энергетический ин-т." (НИУ МЭИ)',
    'Санкт-Петербургский гос. ун-т. телекоммуникаций им. проф. М.А. Бонч-Бруевича': 'Санкт-Петербургский гос. ун-т. телекоммуникаций им. М.А. Бонч-Бруевича',
    'Тывинский гос. ун-т., г. Кызыл': 'Тувинский гос. ун-т., г. Кызыл',
    'Уральский гос. юридический универститет, г. Екатеринбург': 'Уральский гос. юридический ун-т., г. Екатеринбург',
}

uni_map = pd.DataFrame({"university": df["university"].unique()})
uni_map["key"]  = uni_map["university"].map(make_key)
uni_map["abbr"] = uni_map["university"].map(extract_abbr)
cnt = df.groupby("university").agg(n=("year", "size"), last_year=("year", "max"))
uni_map = uni_map.join(cnt, on="university")

# канон = самое частое написание; при равенстве — более позднее, затем алфавит (детерминированно)
canon = (uni_map.sort_values(["key", "n", "last_year", "university"],
                             ascending=[True, False, False, True])
                .drop_duplicates("key").set_index("key")["university"])
uni_map["university_norm"] = uni_map["key"].map(canon)

# ручные слияния поверх авто (значения — канонические названия, без цепочек A→B→C)
assert set(MANUAL.values()) <= set(uni_map["university_norm"]), "значение MANUAL должно быть каноническим"
manual = uni_map["university"].map(MANUAL).astype(uni_map["university_norm"].dtype)
uni_map["university_norm"] = manual.fillna(uni_map["university_norm"])

df["university_norm"] = df["university"].map(uni_map.set_index("university")["university_norm"])
assert df["university_norm"].notna().all()
assert len(df) == 167974
uni_map.to_csv("university_map.csv", sep=";", index=False, encoding="utf-8-sig")

# --- проверки ---
vu = df[df.category == "Данные по вузам"]
print("уникальных названий:", df["university"].nunique(), "→", df["university_norm"].nunique())

# зерно вуз×год×приём: новые дубли = склеили РАЗНЫЕ вузы, надо разбирать
d0 = vu.duplicated(["university", "year", "admission_type"]).sum()
d1 = vu.duplicated(["university_norm", "year", "admission_type"]).sum()
print("дубли зерна: до", d0, "→ после", d1)
if d1 > d0:
    bad = vu[vu.duplicated(["university_norm", "year", "admission_type"], keep=False)]
    print(bad.groupby(["university_norm", "year", "admission_type"])["university"].unique().head(20))

# филиалы не должны смешаться с головными вузами
fil = uni_map.assign(is_fil=uni_map["university"].str.contains("филиал", case=False))
mixed = fil.groupby("university_norm")["is_fil"].nunique()
print("смешались филиалы и головные:", mixed[mixed > 1].index.tolist())

# панель (было 155 по любому приёму); платный 2012–2013 битый → считаем по 13 годам
print("во все 15 лет (любой приём):", (vu.groupby("university_norm")["year"].nunique() == 15).sum())
pb = vu[vu.admission_type == "Бюджетный прием"].groupby("university_norm")["year"].nunique()
pp = vu[(vu.admission_type == "Платный прием") & ~vu.year.isin([2012, 2013])].groupby("university_norm")["year"].nunique()
print("бюджет, 15/15 лет:", (pb == 15).sum(), "| платный, 13/13 лет:", (pp == 13).sum())

уникальных названий: 1393 → 1285
дубли зерна: до 2 → после 2
смешались филиалы и головные: []
во все 15 лет (любой приём): 199
бюджет, 15/15 лет: 164 | платный, 13/13 лет: 123


In [12]:
# 7b. Ревизия автосклейки + разбор исходных дублей
merged = (uni_map.groupby("university_norm")["university"]
                 .agg(list).loc[lambda s: s.map(len) > 1])
print("групп склеено:", len(merged), "| названий в них:", merged.map(len).sum())
pd.DataFrame({"canonical": merged.index, "variants": merged.values}).to_csv(
    "name_merges_auto.csv", sep=";", index=False, encoding="utf-8-sig")
for norm, names in merged.head(15).items():
    print("→", norm)
    for n in names:
        print("   ", n)

vu = df[df.category == "Данные по вузам"]
dup = vu[vu.duplicated(["university", "year", "admission_type"], keep=False)]
print("\nисходные дубли зерна:")
print(dup[["university", "year", "admission_type", "avg_score", "enrolled", "source_file"]].to_string(index=False))

групп склеено: 90 | названий в них: 189
→ Балтийский гос. техн. ун-т. "Военмех" им. Д.Ф. Устинова, г. Санкт-Петербург
    Балтийский гос. техн. ун-т. ВОЕНМЕХ им. Д.Ф. Устинова, г. Санкт-Петербург
    Балтийский гос. техн. ун-т. "Военмех" им. Д.Ф. Устинова, г. Санкт-Петербург
→ Башкирский гос. педагогический ун-т. им. М. Акмуллы, г. Уфа
    Башкирский гос. педагогический ун-т. им. М.Акмуллы, г. Уфа
    Башкирский гос. педагогический ун-т. им. М. Акмуллы, г. Уфа
→ Белгородский гос. национальный исследовательский ун-т.
    Белгородский гос. ун-т. (БелГУ)
    Белгородский гос. национальный исследовательский ун-т.
→ Восточно-Сибирский гос. ун-т. технологий и управления, г. Улан-Удэ
    Восточно-Сибирский гос. ун-т. технологий и управления, г.Улан-Удэ
    Восточно-Сибирский гос. ун-т. технологий и управления, г. Улан-Удэ
→ Выксунский филиал Национального исследовательского технологического ун-т.а "МИСиС"
    Выксунский филиал Национального исследовательского технологического ун-т.а "МИСиС"
 

In [15]:
# 8. Кандидаты v2: фильтр по годам + token_sort + разбор различий
from rapidfuzz import process, fuzz

c = (uni_map.groupby("university_norm")
        .agg(key=("key", "first"), n=("n", "sum"), last_year=("last_year", "max"))
        .reset_index())
c["abbr"]  = c["university_norm"].map(
    uni_map.dropna(subset=["abbr"]).groupby("university_norm")["abbr"].first())
c["years"] = c["university_norm"].map(df.groupby("university_norm")["year"].agg(set))
c["tok"]   = c["key"].str.split().map(set)
c["is_fil"] = c["university_norm"].str.contains("филиал", case=False)

found = {}

def add(i, j, score, why):
    if c.is_fil[i] != c.is_fil[j]:            # филиал ≠ головной вуз
        return
    if len(c.years[i] & c.years[j]) > 1:      # живут в одних годах → разные вузы
        return
    k = tuple(sorted((i, j)))
    if k not in found or found[k][0] < score:
        found[k] = (score, why)

keys = c["key"].tolist()
for i, k in enumerate(keys):
    for _, score, j in process.extract(k, keys, scorer=fuzz.token_sort_ratio,
                                       score_cutoff=85, limit=None):
        if j > i:
            add(i, j, round(score, 1), "текст")

for i, ab in c["abbr"].items():               # (МГИМО) ↔ «мгимо» в другом названии
    if isinstance(ab, str):
        for j, tk in c["tok"].items():
            if j != i and ab.lower() in tk:
                add(i, j, 100.0, "аббревиатура")

rows = []
for (i, j), (score, why) in found.items():
    a, b = c.loc[i], c.loc[j]
    rows.append({"name_a": a.university_norm, "name_b": b.university_norm,
                 "score": score, "why": why,
                 "only_a": " ".join(sorted(a.tok - b.tok)),
                 "only_b": " ".join(sorted(b.tok - a.tok)),
                 "years_a": f"{min(a.years)}–{max(a.years)}",
                 "years_b": f"{min(b.years)}–{max(b.years)}",
                 "n_a": a.n, "n_b": b.n})

cand = (pd.DataFrame(rows)
          .sort_values(["score", "name_a"], ascending=[False, True])
          .reset_index(drop=True))
cand.to_csv("name_candidates.csv", sep=";", index=False, encoding="utf-8-sig")
print(len(cand), "пар после фильтров | из них score ≥ 95:", (cand.score >= 95).sum())

for r in cand[cand.score >= 95].itertuples():
    print(f"\n[{r.Index}] {r.score} {r.why} | {r.years_a} vs {r.years_b}")
    print("   A:", r.name_a)
    print("   B:", r.name_b)
    print("   только A:", r.only_a or "—", "| только B:", r.only_b or "—")

216 пар после фильтров | из них score ≥ 95: 19

[0] 100.0 текст | 2015–2021 vs 2011–2014
   A: Белгородский гос. ин-т. искусств и культуры
   B: Белгородский гос. ин-т. культуры и искусств
   только A: — | только B: —

[1] 100.0 аббревиатура | 2011–2025 vs 2012–2013
   A: Белгородский гос. национальный исследовательский ун-т.
   B: Белгородский гос. национальный исследовательский ун-т. (НИУ «БелГУ»)
   только A: — | только B: белгу

[2] 100.0 аббревиатура | 2018–2025 vs 2025–2025
   A: Бронницкий филиал Московского автомобильно-дорожного государственного технического ун-т.а (МАДИ)
   B: Прикаспийский межрегиональный ин-т. (филиал) МАДИ, г. Махачкала
   только A: автомобильно бронницкий государственного дорожного московского технического университета | только B: институт мади махачкала межрегиональный прикаспийский

[3] 100.0 аббревиатура | 2018–2025 vs 2025–2025
   A: Волжский филиал Московского автомобильно-дорожного государственного технического ун-т.а (МАДИ)
   B: Прикаспийский межр

In [19]:
# 8b. Подтверждённые пары → готовый текст для MANUAL в ячейке 7
KEEP = [0, 1, 8, 9, 10, 12, 13, 16, 18]          # номера строк из вывода выше, например [0, 2, 5]

n_by    = c.set_index("university_norm")["n"].to_dict()
last_by = c.set_index("university_norm")["last_year"].to_dict()
manual_new, winners, losers = {}, set(), set()

for i in KEEP:
    a, b = cand.loc[i, "name_a"], cand.loc[i, "name_b"]
    win, lose = (a, b) if (n_by[a], last_by[a]) >= (n_by[b], last_by[b]) else (b, a)
    winners.add(win); losers.add(lose)
    for s in uni_map.loc[uni_map.university_norm == lose, "university"]:
        if s not in MANUAL:
            manual_new[s] = win

assert not (winners & losers), "цепочка A→B→C: убери противоречивые пары из KEEP"
print("# добавь эти строки в MANUAL ячейки 7:")
for k, v in sorted(manual_new.items()):
    print(f"    {k!r}: {v!r},")

# добавь эти строки в MANUAL ячейки 7:
    'Белгородский гос. ин-т. культуры и искусств': 'Белгородский гос. ин-т. искусств и культуры',
    'Белгородский гос. национальный исследовательский ун-т. (НИУ «БелГУ»)': 'Белгородский гос. национальный исследовательский ун-т.',
    'Моск. гос. гуманитарный экономический ун-т.': 'Моск. гос. гуманитарно-экономический ун-т.',
    'Моск. гос. машиностроительный ун-т. "МАМИ" (бывший Моск. гос. техн. ун-т. "МАМИ")': 'Моск. гос. машиностроительный ун-т. (МАМИ)',
    'Моск. гос. техн. ун-т. "МАМИ"': 'Моск. гос. машиностроительный ун-т. (МАМИ)',
    'Моск. энергетический ин-т. (МЭИ)': 'Национальный исследовательский ун-т. "Моск. энергетический ин-т." (НИУ МЭИ)',
    'Санкт-Петербургский гос. ун-т. телекоммуникаций им. проф. М.А. Бонч-Бруевича': 'Санкт-Петербургский гос. ун-т. телекоммуникаций им. М.А. Бонч-Бруевича',
    'Тывинский гос. ун-т., г. Кызыл': 'Тувинский гос. ун-т., г. Кызыл',
    'Уральский гос. юридический универститет, г. Екатеринбург': '